In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving Training Dataset.arff to Training Dataset.arff


In [ ]:
!pip install scikeras

In [ ]:
from scipy.io import arff
import pandas as pd

# Chargement du fichier ARFF
data = arff.loadarff("Training Dataset.arff")
df = pd.DataFrame(data[0])  # Conversion en DataFrame

# 2. Conversion des colonnes catégorielles (bytes -> str)
for col in df.columns:
    if df[col].dtype == object:  # Si la colonne est de type object (bytes)
        df[col] = df[col].str.decode('utf-8')  # Conversion bytes -> str

# Import necessary libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib import cm
import warnings
warnings.filterwarnings('ignore')

# For data preprocessing
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler, PowerTransformer
from sklearn.metrics import (classification_report, roc_curve, auc, accuracy_score,
                            precision_recall_curve, confusion_matrix, average_precision_score,
                            f1_score, precision_score, recall_score)
from sklearn.feature_selection import RFECV, mutual_info_classif
from sklearn.ensemble import IsolationForest
from sklearn.decomposition import PCA

# For imbalanced data handling
from imblearn.over_sampling import SMOTE

# Deep Learning libraries
import tensorflow as tf
from tensorflow.keras.models import Sequential, Model, load_model
from tensorflow.keras.layers import (Dense, Dropout, BatchNormalization, LSTM, Bidirectional,
                                    Activation, Input, Reshape, TimeDistributed,
                                    MaxPooling1D, Flatten, GlobalAveragePooling1D)
from tensorflow.keras.callbacks import (EarlyStopping, ModelCheckpoint,
                                       ReduceLROnPlateau, TensorBoard)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils import plot_model
import pickle

# Set display precision & style
np.set_printoptions(precision=4)
pd.set_option('display.precision', 4)
sns.set_style('whitegrid')
plt.style.use('seaborn-v0_8-pastel')

# ---------------------------
# 0. Loading and exploring the dataset
# ---------------------------
print("="*80)
print("PHISHING DETECTION WITH LSTM - ADVANCED ANALYSIS")
print("="*80)

print(f"Dataset dimensions: {df.shape[0]} rows × {df.shape[1]} columns")

# Create a figure with two subplots for initial exploration
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 6))

# Display class distribution
target_column = 'Result' if 'Result' in df.columns else 'label'
class_counts = df[target_column].value_counts()
colors = ['#6495ED', '#FF7F50']
ax1.pie(class_counts, labels=class_counts.index, autopct='%1.1f%%',
        colors=colors, shadow=True, startangle=90, explode=(0.1, 0))
ax1.set_title('Class Distribution', fontsize=14, pad=20)

for col in df.columns:
    if col != target_column:
        df[col] = pd.to_numeric(df[col], errors='coerce')
numeric_columns = df.select_dtypes(include=['int64', 'float64']).columns[:10]
print(f"Colonnes numériques détectées : {list(numeric_columns)}")
stats_df = df[numeric_columns].describe().T[['mean', 'std', 'min', 'max']]
sns.heatmap(stats_df, cmap='coolwarm', annot=True, fmt='.2f', ax=ax2)
ax2.set_title('Descriptive Statistics (First 10 variables)', fontsize=14, pad=20)

plt.tight_layout()
plt.savefig('class_distribution.png')
plt.close()

# ---------------------------
# 1. Advanced Data Preparation
# ---------------------------
print("\n" + "="*80)
print("DATA PREPARATION AND PREPROCESSING")
print("="*80)

# Remove non-relevant columns (e.g., 'id' and 'url')
for col in ['id', 'url']:
    if col in df.columns:
        df.drop(columns=[col], inplace=True)

# Check and handle missing values
missing_values = df.isnull().sum()
print("\nMissing values per column:")
print(missing_values[missing_values > 0] if missing_values.sum() > 0 else "No missing values")

# Remove observations with missing values in the target (if necessary)
df.dropna(subset=[target_column], inplace=True)

# Check correlations between variables
corr_matrix = df.corr()
plt.figure(figsize=(16, 12))
mask = np.zeros_like(corr_matrix, dtype=bool)
mask[np.triu_indices_from(mask)] = True
cmap = sns.diverging_palette(230, 20, as_cmap=True)
sns.heatmap(corr_matrix, mask=mask, cmap=cmap, vmax=1, vmin=-1, center=0,
            square=True, linewidths=.5, annot=False, cbar_kws={'shrink': .5})
plt.title('Correlation Matrix of Variables', fontsize=16, pad=20)
plt.tight_layout()
plt.savefig('correlation_matrix.png')
plt.close()

# Separate features and target
X = df.drop(columns=[target_column])
y = df[target_column]
feature_names = X.columns

# Analysis of initial features
print(f"\nNumber of initial features: {X.shape[1]}")
print(f"Initial class distribution: {y.value_counts().to_dict()}")

# ---------------------------
# 2. Advanced preprocessing: Scaling with PowerTransformer + StandardScaler
# ---------------------------
# PowerTransformer to normalize distributions
print("\nApplying transformations to normalize distributions...")
power_transformer = PowerTransformer(method='yeo-johnson')
X_transformed = power_transformer.fit_transform(X)

# Standard Scaler to center and reduce
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_transformed)
print("Transformations applied.")

# ---------------------------
# 3. Anomaly detection and removal with IsolationForest
# ---------------------------
print("\nAnomaly detection with IsolationForest:")
chosen_contamination = 0.01
iso = IsolationForest(contamination=chosen_contamination, random_state=42, n_jobs=-1)
pred_anomalies = iso.fit_predict(X_scaled)
mask = pred_anomalies != -1
X_clean = X_scaled[mask]
y_clean = y[mask]

# Visualize anomalies (PCA for 2D visualization)
# PCA pour 2D
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

# Assure-toi que y est bien numérique
y = pd.Series(y).astype(int).values  # <- ligne importante

# Visualisation
plt.figure(figsize=(10, 8))
plt.scatter(X_pca[mask, 0], X_pca[mask, 1], c=y[mask], cmap='viridis',
            alpha=0.6, s=30, edgecolor='k', linewidth=0.3, label='Normal')
plt.scatter(X_pca[~mask, 0], X_pca[~mask, 1], c='red', marker='x',
            s=50, label='Anomalies')
plt.title('Visualization of Detected Anomalies (PCA)', fontsize=14)
plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')
plt.legend()
plt.savefig('anomalies_pca.png')
plt.close()

anomaly_count = len(X_scaled) - np.sum(mask)
print(f"Points removed as anomalies: {anomaly_count} ({anomaly_count/len(X_scaled)*100:.2f}%)")

# ---------------------------
# 4. Class balancing with SMOTE
# ---------------------------
print("\nBalancing classes with SMOTE...")
smote = SMOTE(random_state=42)
X_res, y_res = smote.fit_resample(X_clean, y_clean)

# Visualization before/after balancing
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Before SMOTE
ax1.bar(['Legitimate', 'Phishing'], [np.sum(y_clean == 0), np.sum(y_clean == 1)],
        color=['#6495ED', '#FF7F50'])
ax1.set_title('Class Distribution - Before SMOTE', fontsize=14)
ax1.set_ylabel('Number of samples')

# After SMOTE
ax2.bar(['Legitimate', 'Phishing'], [np.sum(y_res == 0), np.sum(y_res == 1)],
        color=['#6495ED', '#FF7F50'])
ax2.set_title('Class Distribution - After SMOTE', fontsize=14)
ax2.set_ylabel('Number of samples')

plt.tight_layout()
plt.savefig('smote_balancing.png')
plt.close()

from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
y_encoded = le.fit_transform(y)  # y devient 0 et 1

# SMOTE avec les bons labels
smote = SMOTE(random_state=42)
print("Valeurs uniques dans y_res :", np.unique(y_res))

# ---------------------------
# 5. RFECV for optimal feature selection
# ---------------------------
print("\n" + "="*80)
print("FEATURE SELECTION WITH RFECV")
print("="*80)

# Use a simple model as the basis for RFECV
from sklearn.ensemble import RandomForestClassifier
base_clf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)

# RFECV configuration for optimal feature selection
rfecv = RFECV(
    estimator=base_clf,
    step=1,  # Eliminate 1 feature at each step
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    scoring='f1',  # F1 score as metric (well adapted to imbalanced problems)
    n_jobs=-1,  # Use all available processors
    verbose=1,
    min_features_to_select=5  # Keep at least 5 features
)
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y_res = le.fit_transform(y_res)  # Cela garantit que les labels sont transformés en [0, 1] ou [-1, 1]
print(np.unique(y_res))  # Vérifiez que cela affiche [0, 1] ou [-1, 1]
print("\nRunning feature selection with RFECV (please be patient, this may take time)...")
rfecv.fit(X_res, y_res)

# RFECV results
print(f"\nOptimal number of features: {rfecv.n_features_}")
if rfecv.n_features_ <= len(rfecv.cv_results_['mean_test_score']):
    print(f"Score with {rfecv.n_features_} features: {rfecv.cv_results_['mean_test_score'][rfecv.n_features_ - 1]:.4f}")
else:
    last_idx = len(rfecv.cv_results_['mean_test_score']) - 1
    print(f"Score with {last_idx + 1} features: {rfecv.cv_results_['mean_test_score'][last_idx]:.4f} (adjusted)")
# RFECV scores visualization
plt.figure(figsize=(12, 6))
plt.plot(range(1, len(rfecv.cv_results_['mean_test_score']) + 1), rfecv.cv_results_['mean_test_score'], marker='o',
         color='#6495ED', linestyle='-', linewidth=2, markersize=8)
plt.axvline(x=rfecv.n_features_, color='#FF7F50', linestyle='--', label=f'Optimal: {rfecv.n_features_} features')
plt.fill_between(range(1, len(rfecv.cv_results_['mean_test_score']) + 1), rfecv.cv_results_['mean_test_score'] - np.std(rfecv.cv_results_['mean_test_score']),
                rfecv.cv_results_['mean_test_score'] + np.std(rfecv.cv_results_['mean_test_score']), alpha=0.2, color='#6495ED')
plt.xlabel('Number of Features', fontsize=12)
plt.ylabel('Cross-Validation Score (F1)', fontsize=12)
plt.title('Cross-Validation Scores vs Number of Features', fontsize=14)
plt.grid(True, linestyle='--', alpha=0.7)
plt.legend()
plt.tight_layout()
plt.savefig('rfecv_scores.png')
plt.close()

# Get selected features
selected_features_mask = rfecv.support_
selected_feature_names = feature_names[selected_features_mask]

# Display selected features
print("\nFeatures selected by RFECV:")
for i, feature in enumerate(selected_feature_names):
    print(f"{i+1}. {feature}")

# Transform the data to keep only selected features
X_selected = rfecv.transform(X_res)

# ---------------------------
# 6. Feature importance analysis
# ---------------------------
# Calculate feature importance via mutual information
mi_scores = mutual_info_classif(X_selected, y_res, random_state=42)
mi_df = pd.DataFrame({'Feature': selected_feature_names, 'Importance': mi_scores})
mi_df = mi_df.sort_values('Importance', ascending=False)

plt.figure(figsize=(12, 8))
sns.barplot(x='Importance', y='Feature', data=mi_df, palette='viridis')
plt.title('Feature Importance (Mutual Information)', fontsize=14)
plt.tight_layout()
plt.savefig('feature_importance.png')
plt.close()

# ---------------------------
# 7. Train/test split
# ---------------------------
print("\n" + "="*80)
print("LSTM MODEL TRAINING AND OPTIMIZATION")
print("="*80)

X_train, X_test, y_train, y_test = train_test_split(
    X_selected, y_res, test_size=0.2, random_state=42, stratify=y_res
)

print(f"Training set dimensions: {X_train.shape}")
print(f"Test set dimensions: {X_test.shape}")

# ---------------------------
# 8. LSTM Model Definition
# ---------------------------
# Reshape data for LSTM (LSTM expects 3D input: [samples, timesteps, features])
# We can consider each feature as a time step
X_train_lstm = X_train.reshape(X_train.shape[0], X_train.shape[1], 1)
X_test_lstm = X_test.reshape(X_test.shape[0], X_test.shape[1], 1)

print(f"LSTM input shape: {X_train_lstm.shape}")

# Define the LSTM model
def create_lstm_model(input_shape, learning_rate=0.001):
    model = Sequential([
        # First LSTM layer
        Bidirectional(LSTM(64, return_sequences=True, activation='tanh'), input_shape=input_shape),
        BatchNormalization(),
        Dropout(0.3),

        # Second LSTM layer
        Bidirectional(LSTM(128, return_sequences=True, activation='tanh')),
        BatchNormalization(),
        Dropout(0.4),

        # Third LSTM layer
        Bidirectional(LSTM(64, return_sequences=False, activation='tanh')),
        BatchNormalization(),
        Dropout(0.3),

        # Dense layers
        Dense(64, activation='relu'),
        BatchNormalization(),
        Dropout(0.3),

        # Output layer
        Dense(1, activation='sigmoid')
    ])

    # Compile the model
    model.compile(
        optimizer=Adam(learning_rate=learning_rate),
        loss='binary_crossentropy',
        metrics=['accuracy',
                 tf.keras.metrics.Precision(),
                 tf.keras.metrics.Recall(),
                 tf.keras.metrics.AUC()]
    )

    return model

# Create the model
input_shape = (X_train_lstm.shape[1], 1)
lstm_model = create_lstm_model(input_shape)

# Model summary
print("\nLSTM Model Architecture:")
lstm_model.summary()

# ---------------------------
# 9. Model Training with callbacks
# ---------------------------
# Define callbacks for better training
callbacks = [
    EarlyStopping(
        monitor='val_loss',
        patience=10,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=5,
        min_lr=0.00001,
        verbose=1
    ),
    ModelCheckpoint(
        'best_lstm_phishing_model.h5',
        monitor='val_loss',
        save_best_only=True,
        verbose=1
    )
]

# Train the model
print("\nTraining the LSTM model...")
history = lstm_model.fit(
    X_train_lstm, y_train,
    validation_split=0.2,
    epochs=50,
    batch_size=64,
    callbacks=callbacks,
    verbose=1
)

# ---------------------------
# 10. Learning Curves Visualization
# ---------------------------
# Plot training history
plt.figure(figsize=(15, 10))

# Accuracy subplot
plt.subplot(2, 2, 1)
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Model Accuracy', fontsize=14)
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Accuracy', fontsize=12)
plt.legend()
plt.grid(True, alpha=0.3)

# Loss subplot
plt.subplot(2, 2, 2)
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Model Loss', fontsize=14)
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Loss', fontsize=12)
plt.legend()
plt.grid(True, alpha=0.3)

# Precision subplot
plt.subplot(2, 2, 3)
plt.plot(history.history['precision'], label='Training Precision')
plt.plot(history.history['val_precision'], label='Validation Precision')
plt.title('Model Precision', fontsize=14)
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Precision', fontsize=12)
plt.legend()
plt.grid(True, alpha=0.3)

# Recall subplot
plt.subplot(2, 2, 4)
plt.plot(history.history['recall'], label='Training Recall')
plt.plot(history.history['val_recall'], label='Validation Recall')
plt.title('Model Recall', fontsize=14)
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Recall', fontsize=12)
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('lstm_learning_curves.png')
plt.close()

# ---------------------------
# 11. Model Evaluation
# ---------------------------
print("\n" + "="*80)
print("PERFORMANCE EVALUATION AND VISUALIZATION")
print("="*80)

# Evaluate the model on the test set
test_results = lstm_model.evaluate(X_test_lstm, y_test, verbose=0)
print(f"\nTest Loss: {test_results[0]:.4f}")
print(f"Test Accuracy: {test_results[1]:.4f}")
print(f"Test Precision: {test_results[2]:.4f}")
print(f"Test Recall: {test_results[3]:.4f}")
print(f"Test AUC: {test_results[4]:.4f}")

# Get predictions
y_prob = lstm_model.predict(X_test_lstm).flatten()
y_pred = (y_prob >= 0.5).astype(int)

# Classification report
print("\n--- Classification Report ---")
report = classification_report(y_test, y_pred, digits=4)
print(report)

# Confusion Matrix
plt.figure(figsize=(8, 7))
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False)
plt.title('Confusion Matrix', fontsize=14)
plt.ylabel('True Label', fontsize=12)
plt.xlabel('Predicted Label', fontsize=12)
plt.tight_layout()
plt.savefig('lstm_confusion_matrix.png')
plt.close()

# ROC and PR curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 7))

# ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_prob)
roc_auc = auc(fpr, tpr)
ax1.plot(fpr, tpr, color='darkorange', lw=2, label=f'AUC = {roc_auc:.4f}')
ax1.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
ax1.set_xlim([0.0, 1.0])
ax1.set_ylim([0.0, 1.05])
ax1.set_xlabel('False Positive Rate', fontsize=12)
ax1.set_ylabel('True Positive Rate', fontsize=12)
ax1.set_title('ROC Curve', fontsize=14)
ax1.legend(loc="lower right")
ax1.grid(True, alpha=0.3)

# Precision-Recall Curve
precision, recall, _ = precision_recall_curve(y_test, y_prob)
pr_auc = average_precision_score(y_test, y_prob)
ax2.plot(recall, precision, color='blue', lw=2, label=f'AP = {pr_auc:.4f}')
ax2.set_xlim([0.0, 1.0])
ax2.set_ylim([0.0, 1.05])
ax2.set_xlabel('Recall', fontsize=12)
ax2.set_ylabel('Precision', fontsize=12)
ax2.set_title('Precision-Recall Curve', fontsize=14)
ax2.legend(loc="lower left")
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('lstm_roc_pr_curves.png')
plt.close()

# ---------------------------
# 12. Error Analysis
# ---------------------------
# Indices where predictions are wrong
error_indices = np.where(y_pred != y_test)[0]

if len(error_indices) > 0:
    print(f"\nAnalysis of the first {min(10, len(error_indices))} errors:")

    for i, idx in enumerate(error_indices[:10]):
        print(f"Example {i+1}:")
        # Convert y_test and y_pred to numpy array if not already done
        y_test_array = np.array(y_test) if not isinstance(y_test, np.ndarray) else y_test
        y_pred_array = np.array(y_pred) if not isinstance(y_pred, np.ndarray) else y_pred

        print(f"  Prediction: {y_pred_array[idx]}, True label: {y_test_array[idx]}")
        print(f"  Predicted probability: {y_prob[idx]:.4f}")
        print()

# ---------------------------
# 13. Feature Visualization with tSNE
# ---------------------------
# Extract features from the last LSTM layer for visualization
feature_extractor = Model(
    inputs=lstm_model.inputs,
    outputs=lstm_model.layers[-3].output  # Adjust this index based on your final model architecture
)

# Extract features
lstm_features = feature_extractor.predict(X_test_lstm)

# Apply t-SNE for visualization
from sklearn.manifold import TSNE
tsne = TSNE(n_components=2, random_state=42, perplexity=30)
tsne_results = tsne.fit_transform(lstm_features)

# Create a scatter plot colored by classes
plt.figure(figsize=(12, 10))
scatter = plt.scatter(tsne_results[:, 0], tsne_results[:, 1],
                      c=y_test, cmap='coolwarm', s=50, alpha=0.6, edgecolor='k', linewidth=0.3)

# Mark prediction errors
plt.scatter(tsne_results[error_indices, 0], tsne_results[error_indices, 1],
            marker='x', s=100, color='black', label='Errors')

plt.title('t-SNE Visualization of LSTM Features', fontsize=14)
plt.xlabel('t-SNE Component 1', fontsize=12)
plt.ylabel('t-SNE Component 2', fontsize=12)
plt.legend(*scatter.legend_elements(), title="Classes")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('lstm_tsne_features.png')
plt.close()

# ---------------------------
# 14. Model and Result Saving
# ---------------------------
print("\nSaving the LSTM model...")
lstm_model.save('lstm_phishing_model.h5')
print("Model saved as 'lstm_phishing_model.h5'")

# Calculate final metrics
final_metrics = {
    'confusion_matrix': confusion_matrix(y_test, y_pred),
    'accuracy': accuracy_score(y_test, y_pred),
    'precision': precision_score(y_test, y_pred),
    'recall': recall_score(y_test, y_pred),
    'f1': f1_score(y_test, y_pred),
    'fpr': fpr,
    'tpr': tpr,
    'roc_auc': roc_auc,
    'precision_curve': precision,
    'recall_curve': recall,
    'pr_auc': pr_auc,
    'test_results': test_results
}

# Save model, preprocessors, and metrics
model_data = {
    'power_transformer': power_transformer,
    'scaler': scaler,
    'selected_features_mask': selected_features_mask,
    'selected_feature_names': selected_feature_names,
    'feature_names': feature_names,
    'metrics': final_metrics,
    'input_shape': input_shape
}

with open('lstm_phishing_model_data.pkl', 'wb') as file:
    pickle.dump(model_data, file)

print("\nMetrics and preprocessors saved as 'lstm_phishing_model_data.pkl'")

# Compare LSTM with CNN model metrics
if False:  # Set to True if you have already run the CNN model and have its metrics
    try:
        with open('cnn_phishing_model_data.pkl', 'rb') as file:
            cnn_data = pickle.load(file)

        # Create comparison plot
        plt.figure(figsize=(12, 8))

        metrics = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']
        lstm_scores = [final_metrics[m] for m in metrics]
        cnn_scores = [cnn_data['metrics'][m] for m in metrics]

        x = np.arange(len(metrics))
        width = 0.35

        plt.bar(x - width/2, lstm_scores, width, label='LSTM', color='#6495ED')
        plt.bar(x + width/2, cnn_scores, width, label='CNN', color='#FF7F50')

        plt.xlabel('Metric', fontsize=12)
        plt.ylabel('Score', fontsize=12)
        plt.title('LSTM vs CNN Performance Comparison', fontsize=14)
        plt.xticks(x, metrics)
        plt.ylim(0, 1)
        plt.legend()
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.savefig('model_comparison.png')
        plt.close()

        print("\nModel comparison completed and saved.")
    except:
        print("\nCould not load CNN model data for comparison.")

print("\n" + "="*80)
print("ANALYSIS COMPLETED")
print("="*80)

PHISHING DETECTION WITH LSTM - ADVANCED ANALYSIS
Dataset dimensions: 11055 rows × 31 columns
Colonnes numériques détectées : ['having_IP_Address', 'URL_Length', 'Shortining_Service', 'having_At_Symbol', 'double_slash_redirecting', 'Prefix_Suffix', 'having_Sub_Domain', 'SSLfinal_State', 'Domain_registeration_length', 'Favicon']

DATA PREPARATION AND PREPROCESSING

Missing values per column:
No missing values

Number of initial features: 30
Initial class distribution: {'1': 6157, '-1': 4898}

Applying transformations to normalize distributions...
Transformations applied.

Anomaly detection with IsolationForest:
Points removed as anomalies: 111 (1.00%)

Balancing classes with SMOTE...
Valeurs uniques dans y_res : ['-1' '1']

FEATURE SELECTION WITH RFECV
[0 1]

Running feature selection with RFECV (please be patient, this may take time)...
Fitting estimator with 30 features.
Fitting estimator with 29 features.
Fitting estimator with 28 features.
Fitting estimator with 27 features.
Fitting 

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ bidirectional (Bidirectional)   │ (None, 22, 128)        │        33,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 22, 128)        │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 22, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_1 (Bidirectional) │ (None, 22, 256)        │       263,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 22, 256)        │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 22, 256)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_2 (Bidirectional) │ (None, 128)            │       164,352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 471,937 (1.80 MB)

 Trainable params: 470,785 (1.80 MB)

...

123/123 ━━━━━━━━━━━━━━━━━━━━ 28s 231ms/step - accuracy: 0.9658 - auc: 0.9939 - loss: 0.0913 - precision: 0.9650 - recall: 0.9658 - val_accuracy: 0.9618 - val_auc: 0.9888 - val_loss: 0.1234 - val_precision: 0.9639 - val_recall: 0.9610 - learning_rate: 5.0000e-04
Epoch 27/50
123/123 ━━━━━━━━━━━━━━━━━━━━ 0s 208ms/step - accuracy: 0.9722 - auc: 0.9972 - loss: 0.0692 - precision: 0.9732 - recall: 0.9712
Epoch 27: val_loss improved from 0.12336 to 0.12072, saving model to best_lstm_phishing_model.h5


123/123 ━━━━━━━━━━━━━━━━━━━━ 28s 224ms/step - accuracy: 0.9722 - auc: 0.9972 - loss: 0.0692 - precision: 0.9732 - recall: 0.9712 - val_accuracy: 0.9638 - val_auc: 0.9894 - val_loss: 0.1207 - val_precision: 0.9586 - val_recall: 0.9710 - learning_rate: 5.0000e-04
Epoch 28/50
123/123 ━━━━━━━━━━━━━━━━━━━━ 0s 210ms/step - accuracy: 0.9686 - auc: 0.9959 - loss: 0.0796 - precision: 0.9710 - recall: 0.9662
Epoch 28: val_loss did not improve from 0.12072
123/123 ━━━━━━━━━━━━━━━━━━━━ 41s 225ms/step - accuracy: 0.9686 - auc: 0.9959 - loss: 0.0797 - precision: 0.9710 - recall: 0.9662 - val_accuracy: 0.9562 - val_auc: 0.9866 - val_loss: 0.1390 - val_precision: 0.9378 - val_recall: 0.9790 - learning_rate: 5.0000e-04
Epoch 29/50
123/123 ━━━━━━━━━━━━━━━━━━━━ 0s 208ms/step - accuracy: 0.9672 - auc: 0.9954 - loss: 0.0829 - precision: 0.9718 - recall: 0.9620
Epoch 29: val_loss did not improve from 0.12072
123/123 ━━━━━━━━━━━━━━━━━━━━ 41s 230ms/step - accuracy: 0.9672 - auc: 0.9954 - loss: 0.0829 - precis

123/123 ━━━━━━━━━━━━━━━━━━━━ 40s 226ms/step - accuracy: 0.9706 - auc: 0.9967 - loss: 0.0689 - precision: 0.9711 - recall: 0.9697 - val_accuracy: 0.9618 - val_auc: 0.9909 - val_loss: 0.1199 - val_precision: 0.9557 - val_recall: 0.9700 - learning_rate: 5.0000e-04
Epoch 31/50
123/123 ━━━━━━━━━━━━━━━━━━━━ 0s 209ms/step - accuracy: 0.9739 - auc: 0.9975 - loss: 0.0644 - precision: 0.9771 - recall: 0.9698
Epoch 31: val_loss improved from 0.11988 to 0.11897, saving model to best_lstm_phishing_model.h5


123/123 ━━━━━━━━━━━━━━━━━━━━ 28s 224ms/step - accuracy: 0.9739 - auc: 0.9975 - loss: 0.0644 - precision: 0.9771 - recall: 0.9698 - val_accuracy: 0.9633 - val_auc: 0.9904 - val_loss: 0.1190 - val_precision: 0.9631 - val_recall: 0.9650 - learning_rate: 5.0000e-04
Epoch 32/50
123/123 ━━━━━━━━━━━━━━━━━━━━ 0s 213ms/step - accuracy: 0.9709 - auc: 0.9966 - loss: 0.0721 - precision: 0.9737 - recall: 0.9665
Epoch 32: val_loss did not improve from 0.11897
123/123 ━━━━━━━━━━━━━━━━━━━━ 28s 228ms/step - accuracy: 0.9709 - auc: 0.9966 - loss: 0.0721 - precision: 0.9737 - recall: 0.9665 - val_accuracy: 0.9572 - val_auc: 0.9867 - val_loss: 0.1340 - val_precision: 0.9439 - val_recall: 0.9740 - learning_rate: 5.0000e-04
Epoch 33/50
123/123 ━━━━━━━━━━━━━━━━━━━━ 0s 211ms/step - accuracy: 0.9736 - auc: 0.9976 - loss: 0.0618 - precision: 0.9733 - recall: 0.9742
Epoch 33: val_loss did not improve from 0.11897
123/123 ━━━━━━━━━━━━━━━━━━━━ 29s 232ms/step - accuracy: 0.9735 - auc: 0.9976 - loss: 0.0619 - precis


Saving the LSTM model...
Model saved as 'lstm_phishing_model.h5'

Metrics and preprocessors saved as 'lstm_phishing_model_data.pkl'

ANALYSIS COMPLETED
